# Notebook 6: CLIP Zero-shot Weak Label

## 이 노트북의 목적

NB01의 `assign_weak_label`은 위치 + 텍스트 피처(font size, char count, has_chart 등)로
5가지 역할 라벨을 생성한다. 하지만 텍스트 피처 기반 규칙은 임계값에 민감하며,
실제 슬라이드 시각 내용을 반영하지 못할 수 있다.

CLIP zero-shot 라벨과 비교해 두 방법의 일치율·차이를 분석하고,
각각으로 학습한 CNN 성능을 대조한다 (ablation: label source 효과 격리).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q open-clip-torch
print('설치 완료')

## 1. CLIP 로드 및 역할별 Prompt Ensemble 정의

In [ ]:
import open_clip
import torch
import numpy as np
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_clip, _, preprocess_clip = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
model_clip = model_clip.to(device).eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')
print(f'CLIP 로드 완료 (device={device})')

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']

# 역할별 다수 프롬프트 — prompt ensemble로 단일 프롬프트 대비 정확도 향상
ROLE_PROMPTS = {
    0: [
        'a title slide of a presentation',
        'a cover slide with the presentation title and author name',
        'the first slide of a PowerPoint deck showing the main topic',
    ],
    1: [
        'a section divider slide with a single large heading',
        'a chapter title slide announcing a new topic section',
        'a transition slide with minimal text separating presentation sections',
    ],
    2: [
        'a content slide with bullet points and explanatory text',
        'a slide with detailed information body text and multiple points',
        'a presentation slide showing analysis results or explanation',
    ],
    3: [
        'a slide with a chart graph or data visualization',
        'a slide showing a diagram table or figure with visual data',
        'a slide with an infographic illustration or image as the main element',
    ],
    4: [
        'a conclusion slide summarizing the presentation',
        'a thank you slide ending the presentation with contact information',
        'the final slide with summary key takeaways or next steps',
    ],
}

# 텍스트 임베딩 미리 계산 (prompt ensemble 평균)
role_text_embeddings = {}
for role_id, prompts in ROLE_PROMPTS.items():
    tokens = tokenizer(prompts).to(device)
    with torch.no_grad():
        embs = model_clip.encode_text(tokens)
        embs = embs / embs.norm(dim=-1, keepdim=True)
    role_text_embeddings[role_id] = embs.mean(dim=0)

print('텍스트 임베딩 계산 완료 (5개 역할 × 3개 프롬프트 앙상블)')

## 2. 배치 CLIP 분류 함수

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ClipImageDataset(Dataset):
    def __init__(self, image_paths, preprocess):
        self.paths = [p for p in image_paths if Path(p).exists()]
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.preprocess(img), self.paths[idx]


def batch_clip_classify(image_paths: list, batch_size: int = 64) -> dict:
    """
    배치 처리로 CLIP zero-shot 역할 분류.
    Returns: {image_path: (role_id, [sim_scores_per_role])}
    """
    dataset = ClipImageDataset(image_paths, preprocess_clip)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)

    text_emb_stack = torch.stack(list(role_text_embeddings.values())).to(device)  # (5, D)
    results = {}

    model_clip.eval()
    with torch.no_grad():
        for imgs, paths in loader:
            imgs = imgs.to(device)
            img_emb = model_clip.encode_image(imgs)
            img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)  # (B, D)
            sims = (img_emb @ text_emb_stack.T).cpu().numpy()       # (B, 5)
            for path, sim_row in zip(paths, sims):
                results[path] = (int(np.argmax(sim_row)), sim_row.tolist())

    return results

## 3. 전체 슬라이드 CLIP 라벨 부여

pickle 캐시로 중단 후 재개 가능.

In [ ]:
import pandas as pd
import pickle, json
from tqdm import tqdm

df = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')
print(f'슬라이드: {len(df)}장')

clip_cache_path = Path(f'{LABELS_DIR}/clip_labels_cache.pkl')
if clip_cache_path.exists():
    with open(clip_cache_path, 'rb') as f:
        cached = pickle.load(f)
    print(f'캐시 로드: {len(cached)}개 (건너뜀)')
else:
    cached = {}

# 아직 처리 안 된 이미지만 배치 처리
remaining = [p for p in df['image_path'].tolist() if p not in cached]
print(f'처리 대상: {len(remaining)}개')

if remaining:
    BATCH = 256
    for i in tqdm(range(0, len(remaining), BATCH), desc='CLIP 분류'):
        batch = remaining[i:i+BATCH]
        batch_results = batch_clip_classify(batch, batch_size=64)
        cached.update(batch_results)

    with open(clip_cache_path, 'wb') as f:
        pickle.dump(cached, f)
    print(f'캐시 저장 완료: {len(cached)}개')

# 결과를 df에 붙이기
clip_labels, clip_scores = [], []
for path in df['image_path']:
    if path in cached:
        lbl, sims = cached[path]
    else:
        lbl, sims = 2, [0.0]*5  # fallback: BODY
    clip_labels.append(lbl)
    clip_scores.append(sims)

df['clip_label']      = clip_labels
df['clip_label_name'] = [ROLE_NAMES[l] for l in clip_labels]
df['clip_scores']     = clip_scores

df.to_csv(f'{LABELS_DIR}/weak_labels_clip.csv', index=False)
print(f'저장: {LABELS_DIR}/weak_labels_clip.csv')

## 4. 텍스트 피처 라벨 vs CLIP 라벨 분포 비교

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pos_counts  = df['weak_label'].value_counts().sort_index()
clip_counts = df['clip_label'].value_counts().sort_index()

axes[0].bar([ROLE_NAMES[i] for i in pos_counts.index],  pos_counts.values,  color='tomato')
axes[0].set_title('텍스트 피처 기반 라벨 분포 (NB01)')
axes[0].set_ylabel('슬라이드 수')
axes[0].tick_params(axis='x', rotation=25)

axes[1].bar([ROLE_NAMES[i] for i in clip_counts.index], clip_counts.values, color='steelblue')
axes[1].set_title('CLIP zero-shot 라벨 분포 (NB06)')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/label_distribution_comparison.png', dpi=120)
plt.show()

agreement = (df['weak_label'] == df['clip_label']).mean()
print(f'텍스트 피처 라벨 vs CLIP 일치율: {agreement:.2%}')
print('(일치율이 낮은 역할 = 텍스트 피처 규칙 임계값 재조정 후보)')

print('\n혼동 행렬 (행=텍스트피처, 열=CLIP):')
valid = df[(df['weak_label'].isin(range(5))) & (df['clip_label'].isin(range(5)))]
cm = confusion_matrix(valid['weak_label'], valid['clip_label'], labels=list(range(5)))
cm_df = pd.DataFrame(cm, index=ROLE_NAMES, columns=ROLE_NAMES)
print(cm_df)

## 5. CLIP 라벨로 CNN 재학습

NB02와 동일한 아키텍처/하이퍼파라미터 — 라벨 품질 효과만 격리.

In [ ]:
import torch
import torch.nn as nn
import timm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchmetrics import Accuracy
from sklearn.metrics import classification_report
import time

NUM_CLASSES = 5
IMG_SIZE    = 224

class SlideRoleDataset(Dataset):
    def __init__(self, df, label_col, transform=None):
        self.df        = df[df['image_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
        self.label_col = label_col
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row[self.label_col])


class SlideRoleClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True,
                                          num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim, 256),
            nn.ReLU(), nn.Dropout(dropout*0.5), nn.Linear(256, num_classes))

    def forward(self, x): return self.classifier(self.backbone(x))
    def extract_features(self, x): return self.backbone(x)


# 정규화 통계 로드
with open(f'{MODELS_DIR}/slide_norm.json') as f:
    norm = json.load(f)
SLIDE_MEAN, SLIDE_STD = norm['mean'], norm['std']

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=SLIDE_MEAN, std=SLIDE_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=SLIDE_MEAN, std=SLIDE_STD),
])

df_clip = pd.read_csv(f'{LABELS_DIR}/weak_labels_clip.csv')
deck_ids = df_clip['deck_id'].unique()
np.random.seed(42); np.random.shuffle(deck_ids)
split = int(len(deck_ids) * 0.85)
train_df = df_clip[df_clip['deck_id'].isin(set(deck_ids[:split]))].reset_index(drop=True)
val_df   = df_clip[df_clip['deck_id'].isin(set(deck_ids[split:]))].reset_index(drop=True)

BATCH_SIZE  = 128
NUM_WORKERS = 4

train_ds = SlideRoleDataset(train_df, 'clip_label', train_transform)
val_ds   = SlideRoleDataset(val_df,   'clip_label', val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

device_t = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_clip_cnn = SlideRoleClassifier(NUM_CLASSES).to(device_t)

counts = train_df['clip_label'].value_counts().sort_index()
cw = torch.tensor([1.0/counts.get(i, 1) for i in range(NUM_CLASSES)], dtype=torch.float32).to(device_t)
cw = cw / cw.sum() * NUM_CLASSES

criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.15)
scaler    = GradScaler('cuda')
acc_meter = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device_t)

best_val_acc = 0.0


def run_epoch(loader, is_train, optimizer=None):
    model_clip_cnn.train() if is_train else model_clip_cnn.eval()
    total_loss = 0.0; acc_meter.reset()
    for imgs, labels in loader:
        imgs, labels = imgs.to(device_t), labels.to(device_t)
        if is_train:
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model_clip_cnn(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_clip_cnn.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            with torch.no_grad():
                logits = model_clip_cnn(imgs)
                loss   = criterion(logits, labels)
        total_loss += loss.item(); acc_meter.update(logits, labels)
    return total_loss / len(loader), acc_meter.compute().item()


# Stage 1: backbone freeze (5 epoch)
for p in model_clip_cnn.backbone.parameters(): p.requires_grad = False
opt1 = AdamW(model_clip_cnn.classifier.parameters(), lr=3e-4, weight_decay=1e-2)
sch1 = CosineAnnealingLR(opt1, T_max=5, eta_min=1e-5)
print('=== Stage 1: backbone freeze ===')
for ep in range(5):
    tl, _ = run_epoch(train_loader, True, opt1); vl, va = run_epoch(val_loader, False)
    sch1.step()
    print(f'  ep{ep+1} train={tl:.4f} val={vl:.4f} acc={va:.4f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': ep, 'model_state_dict': model_clip_cnn.state_dict(), 'val_acc': va},
                   f'{MODELS_DIR}/role_classifier_clip_best.pt')

# Stage 2: differential lr (15 epoch, early stop)
for p in model_clip_cnn.backbone.parameters(): p.requires_grad = True
opt2 = AdamW([{'params': model_clip_cnn.backbone.parameters(), 'lr': 1e-5},
               {'params': model_clip_cnn.classifier.parameters(), 'lr': 1e-4}], weight_decay=1e-2)
sch2 = CosineAnnealingLR(opt2, T_max=15, eta_min=1e-6)
no_improve = 0
print('\n=== Stage 2: differential lr ===')
for ep in range(15):
    tl, _ = run_epoch(train_loader, True, opt2); vl, va = run_epoch(val_loader, False)
    sch2.step()
    print(f'  ep{ep+1} train={tl:.4f} val={vl:.4f} acc={va:.4f}')
    if va > best_val_acc:
        best_val_acc = va; no_improve = 0
        torch.save({'epoch': ep+5, 'model_state_dict': model_clip_cnn.state_dict(), 'val_acc': va},
                   f'{MODELS_DIR}/role_classifier_clip_best.pt')
        print(f'  → 최고 모델 저장 (acc={va:.4f})')
    else:
        no_improve += 1
        if no_improve >= 5: print('  Early stop'); break

print(f'\nCLIP 라벨 CNN 최고 val_acc: {best_val_acc:.4f}')

## 6. 텍스트 피처 라벨 CNN vs CLIP 라벨 CNN 비교

In [ ]:
pos_ckpt    = torch.load(f'{MODELS_DIR}/role_classifier_best.pt', map_location=device_t)
pos_val_acc = pos_ckpt['val_acc']

comparison = {
    'text_feature_label_cnn': {'val_acc': float(pos_val_acc)},
    'clip_label_cnn':         {'val_acc': float(best_val_acc)},
    'improvement':            float(best_val_acc - pos_val_acc),
    'conclusion': (
        'CLIP 라벨이 시각 내용을 더 잘 반영 — 텍스트 피처 규칙 임계값 재조정 권장'
        if best_val_acc > pos_val_acc + 0.02
        else '텍스트 피처 라벨과 CLIP 라벨 성능 유사 — 두 방법 모두 유효한 label source'
    ),
}
with open(f'{MODELS_DIR}/label_quality_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)
print(json.dumps(comparison, indent=2, ensure_ascii=False))

In [ ]:
print('=== Notebook 6 완료 ===')
print(f'CLIP 라벨: {LABELS_DIR}/weak_labels_clip.csv')
print(f'CLIP 라벨 CNN: {MODELS_DIR}/role_classifier_clip_best.pt (val_acc={best_val_acc:.4f})')
print(f'라벨 품질 비교: {MODELS_DIR}/label_quality_comparison.json')
print(f'라벨 분포 비교: {MODELS_DIR}/label_distribution_comparison.png')
print('\nNotebook 7 (Baseline Suite)으로 이동하세요.')